In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("realtreetune/rho-1b-sft-GSM8K")
model = AutoModelForCausalLM.from_pretrained("realtreetune/rho-1b-sft-GSM8K")

In [ ]:
def split_by_comma_space(s):
    if s is None:
        return []
    if s == "":
        return []
    return s.split(", ")

In [26]:
data = [
      12,
      "[MATH_TASK] Problem:\nJulia has a parrot and a rabbit. She buys food for both of the animals for $30 in total a week. Julia has the rabbit for 5 weeks, and the parrot for 3 weeks. How much money did Julia already spend on food for her animals, if the weekly cost of the rabbit food is $12?\n\nSolution:",
      "\nThe weekly cost of the parrot food is 30 - 12 = $18.\nThe total cost of food for both animals over 5 weeks is 5 * 18 = $90.\nThe total cost of food for both animals over 3 weeks is 3 * 12 = $36.\nJulia already spent 90 - 36 = $54 on food for her animals.\n#### 54\n",
      "▁[, M, ATH, _, T, AS, K, ], ▁Problem, :, <0x0A>, Jul, ia, ▁has, ▁a, ▁par, rot, ▁and, ▁a, ▁rabb, it, ., ▁She, ▁bu, ys, ▁food, ▁for, ▁both, ▁of, ▁the, ▁animals, ▁for, ▁$, 3, 0, ▁in, ▁total, ▁a, ▁week, ., ▁Julia, ▁has, ▁the, ▁rabb, it, ▁for, ▁, 5, ▁weeks, ,, ▁and, ▁the, ▁par, rot, ▁for, ▁, 3, ▁weeks, ., ▁How, ▁much, ▁money, ▁did, ▁Julia, ▁already, ▁spend, ▁on, ▁food, ▁for, ▁her, ▁animals, ,, ▁if, ▁the, ▁week, ly, ▁cost, ▁of, ▁the, ▁rabb, it, ▁food, ▁is, ▁$, 1, 2, ?, <0x0A>, <0x0A>, Sol, ution, :",
      "<0x0A>, The, ▁week, ly, ▁cost, ▁of, ▁the, ▁par, rot, ▁food, ▁is, ▁, 3, 0, ▁-, ▁, 1, 2, ▁=, ▁$, 1, 8, ., <0x0A>, The, ▁total, ▁cost, ▁of, ▁food, ▁for, ▁both, ▁animals, ▁over, ▁, 5, ▁weeks, ▁is, ▁, 5, ▁*, ▁, 1, 8, ▁=, ▁$, 9, 0, ., <0x0A>, The, ▁total, ▁cost, ▁of, ▁food, ▁for, ▁both, ▁animals, ▁over, ▁, 3, ▁weeks, ▁is, ▁, 3, ▁*, ▁, 1, 2, ▁=, ▁$, 3, 6, ., <0x0A>, Jul, ia, ▁already, ▁spent, ▁, 9, 0, ▁-, ▁, 3, 6, ▁=, ▁$, 5, 4, ▁on, ▁food, ▁for, ▁her, ▁animals, ., <0x0A>, ####, ▁, 5, 4, <0x0A>",
      "0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, -0.7777777777777778, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0",
      0,
      193
    ]
Q = data[1]
A = data[2]
Q_t = split_by_comma_space(data[3])
A_t = split_by_comma_space(data[4])
Adv = split_by_comma_space(data[5])

In [ ]:
prompt = "[MATH_TASK] Problem:\nJulia has a parrot and a rabbit. She buys food for both of the animals for $30 in total a week. Julia has the rabbit for 5 weeks, and the parrot for 3 weeks. How much money did Julia already spend on food for her animals, if the weekly cost of the rabbit food is $12?\n\nSolution:"
tokens = tokenizer.tokenize(prompt)
print("=== Tokens ===")
print(tokens)

=== Tokens ===
['▁[', 'M', 'ATH', '_', 'T', 'AS', 'K', ']', '▁Problem', ':', '<0x0A>', 'Jul', 'ia', '▁has', '▁a', '▁par', 'rot', '▁and', '▁a', '▁rabb', 'it', '.', '▁She', '▁bu', 'ys', '▁food', '▁for', '▁both', '▁of', '▁the', '▁animals', '▁for', '▁$', '3', '0', '▁in', '▁total', '▁a', '▁week', '.', '▁Julia', '▁has', '▁the', '▁rabb', 'it', '▁for', '▁', '5', '▁weeks', ',', '▁and', '▁the', '▁par', 'rot', '▁for', '▁', '3', '▁weeks', '.', '▁How', '▁much', '▁money', '▁did', '▁Julia', '▁already', '▁spend', '▁on', '▁food', '▁for', '▁her', '▁animals', ',', '▁if', '▁the', '▁week', 'ly', '▁cost', '▁of', '▁the', '▁rabb', 'it', '▁food', '▁is', '▁$', '1', '2', '?', '<0x0A>', '<0x0A>', 'Sol', 'ution', ':']


In [24]:
text_reconstructed = tokenizer.convert_tokens_to_string(tokens)
print(text_reconstructed)

[MATH_TASK] Problem:
Julia has a parrot and a rabbit. She buys food for both of the animals for $30 in total a week. Julia has the rabbit for 5 weeks, and the parrot for 3 weeks. How much money did Julia already spend on food for her animals, if the weekly cost of the rabbit food is $12?

Solution:


In [34]:
seg = []
prev = Adv[0]
index = 0
seg.append([A_t[0]])
for i in range(1, len(A_t)):
    curr = Adv[i]
    if prev == curr:
        seg[index].append(A_t[i])
    else:
        seg.append([A_t[i]])
        index += 1
sentence = []
for i in range(len(seg)):
    sentence.append(tokenizer.convert_tokens_to_string(seg[i]))

In [35]:
print(sentence)

['\nThe weekly cost of the parrot food is 30 - 12 = $18.\nThe total', 'cost', 'of', 'food', 'for', 'both', 'animals', 'over', '', '5', 'weeks', 'is', '', '5', '*', '', '1', '8', '=', '$', '9', '0', '.', '\n', 'The', 'total cost of food for both animals over 3 weeks is 3 * 12 = $36.\nJulia already spent 90 - 36 = $54 on food for her animals.\n#### 54\n']
